Milestone 1 — Setup & Basic Collection

We'll use the same Open-Meteo weather API from Day 2, but build it as a proper reusable tool this time (not just exploratory code).

Task:

1. Write a function fetch_weather(latitude, longitude, timezone="auto") that:
    1. Makes the API request with proper error handling (timeout=, raise_for_status(), wrapped in try/except)
    2. Returns the parsed JSON data, or None if the request fails
2. Call it for 3 different cities of your choice (pick real coordinates)
3. For each city, confirm the request succeeded and print the top-level keys of the response

This function is the foundation — get it solid before we build the data structuring on top of it.

In [66]:
import requests as req
import pandas as pd
from pathlib import Path
import shutil

In [ ]:
def fetch_weather(lat, lon, timezone="auto"):
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
        "timezone": timezone
    }
    try:
        res = req.get("https://api.open-meteo.com/v1/forecast", params=params, timeout=10)
        res.raise_for_status()
        data = res.json()
        return data
    except req.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return None
        
    

In [54]:
rwp_data = fetch_weather(33.6007,73.0679)
sm_data = fetch_weather(-29.6842,-53.8069)
isb_data = fetch_weather(33.7215,73.0433)


data = [rwp_data, sm_data, isb_data]

for d in data:
    if d is not None:
        print("Success!")
        print(d['daily'].keys())
    else:
        print("Request failed.")


Success!
dict_keys(['time', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum'])
Success!
dict_keys(['time', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum'])
Success!
dict_keys(['time', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum'])


Milestone 2 — Structuring Multiple Cities into One DataFrame

1. Write a function weather_to_df(data, city_name) that takes the raw JSON from fetch_weather() and a city name, and returns a clean DataFrame with columns: city, date, temp_max, temp_min, precipitation
2. Call this function for all 3 cities, and combine the results into a single combined DataFrame using pd.concat()
3. Print the combined DataFrame, and confirm its shape makes sense (should be 3 cities × however many forecast days)
4. Handle the edge case: what should happen if data is None (a failed request) — make sure your function doesn't crash, and the failed city is just skipped in the final combined result

In [55]:
def weather_to_df(data, city_names):
    dfs = []

    for d, city in zip(data, city_names):
        if d is None:
            print(f"Skipping {city} — request failed.")
            continue
        raw_data = {
            "city": city,
            "date": pd.to_datetime(d['daily']['time']),
            "temp_max": d['daily']['temperature_2m_max'],
            "temp_min": d['daily']['temperature_2m_min'],
            "precipitation": d['daily']['precipitation_sum']
        }

        dfs.append(pd.DataFrame(raw_data))
    if not dfs:
        return pd.DataFrame()

    return pd.concat(dfs, ignore_index=True)

city_names = ["RWP", "SM", "ISB"]

combined_df = weather_to_df(data, city_names)

print(combined_df)

   city       date  temp_max  temp_min  precipitation
0   RWP 2026-08-14      32.5      24.7           12.5
1   RWP 2026-08-15      31.6      24.3           10.3
2   RWP 2026-08-16      31.9      24.2            4.5
3   RWP 2026-08-17      31.1      24.7            9.2
4   RWP 2026-08-18      30.7      24.4           11.4
5   RWP 2026-08-19      30.0      23.8           17.7
6   RWP 2026-08-20      29.8      23.8            9.0
7    SM 2026-08-13      17.9      13.0            8.9
8    SM 2026-08-14      18.3      13.2            0.3
9    SM 2026-08-15      23.3      13.3            4.1
10   SM 2026-08-16      28.7      17.4           14.3
11   SM 2026-08-17      32.2      19.3            0.0
12   SM 2026-08-18      32.3      18.7            0.0
13   SM 2026-08-19      29.6      17.9            3.3
14  ISB 2026-08-14      31.4      23.9            9.2
15  ISB 2026-08-15      30.9      23.9            5.6
16  ISB 2026-08-16      30.3      23.8            8.4
17  ISB 2026-08-17      29.7

In [56]:
print(combined_df.shape)

(21, 5)


Milestone 3 (final) — Save, Automate, and Wrap Up

1. Save the combined DataFrame to a CSV file weather_report.csv (using .to_csv(), index=False)
2. Add a simple summary print: average temp_max per city, and which city had the most total precipitation
3. Write a README (same conventions as before — mention this project handles real live API data with graceful failure handling, since that's a genuine differentiator worth highlighting)
4. Push to GitHub

In [67]:
folder = Path("data")
folder.mkdir(exist_ok=True)

combined_df.to_csv("weather_report.csv", index=False)

shutil.move(str("weather_report.csv"), str(folder/"weather_report.csv"))


'data\\weather_report.csv'

In [77]:
avg_max_temp_per_city = combined_df.groupby('city')['temp_max'].mean()
most_total_precip_by_city = combined_df.groupby('city')['precipitation'].sum()
print("===Summary===")
print(avg_max_temp_per_city)
print(most_total_precip_by_city.idxmax())

===Summary===
city
ISB    29.771429
RWP    31.085714
SM     26.042857
Name: temp_max, dtype: float64
RWP
